In [1]:
import os
import pathlib
import sys
import time

import pandas as pd
import psutil
import tomli
from image_analysis_3D.featurization_utils.feature_writing_utils import (
    format_morphology_feature_name,
)
from image_analysis_3D.featurization_utils.intensity_utils import (
    measure_3D_intensity_CPU,
)
from image_analysis_3D.featurization_utils.loading_classes import (
    ImageSetLoader,
    ObjectLoader,
)
from image_analysis_3D.featurization_utils.resource_profiling_util import (
    start_profiling,
    stop_profiling,
)
from image_analysis_3D.file_utils.arg_parsing_utils import (
    check_for_missing_args,
    parse_args,
)
from image_analysis_3D.file_utils.notebook_init_utils import (
    bandicoot_check,
    init_notebook,
)

root_dir, in_notebook = init_notebook()
image_base_dir = bandicoot_check(
    pathlib.Path(os.path.expanduser("~/mnt/bandicoot")).resolve(), root_dir
)

In [2]:
if not in_notebook:
    arguments_dict = parse_args()
    patient = arguments_dict["patient"]
    well_fov = arguments_dict["well_fov"]
    channel = arguments_dict["channel"]
    compartment = arguments_dict["compartment"]
    processor_type = arguments_dict["processor_type"]
    input_subparent_name = arguments_dict["input_subparent_name"]
    mask_subparent_name = arguments_dict["mask_subparent_name"]
    output_features_subparent_name = arguments_dict["output_features_subparent_name"]

else:
    well_fov = "C4-2"
    patient = "NF0014_T1"
    channel = "Mito"
    compartment = "Cytoplasm"
    processor_type = "CPU"
    input_subparent_name = "zstack_images"
    mask_subparent_name = "segmentation_masks"
    output_features_subparent_name = "extracted_features"

image_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{input_subparent_name}/{well_fov}/"
)
mask_set_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{mask_subparent_name}/{well_fov}/"
)
output_parent_path = pathlib.Path(
    f"{image_base_dir}/data/{patient}/{output_features_subparent_name}/{well_fov}/"
)
output_parent_path.mkdir(parents=True, exist_ok=True)
channel_mapping_file_path = pathlib.Path(
    f"{root_dir}/config/channel_mapping.toml"
).resolve(strict=True)

In [3]:
# read in channel mapping
with open(channel_mapping_file_path, "rb") as f:
    channel_mapping_dict = tomli.load(f)
channel_n_compartment_mapping = channel_mapping_dict["channel_mapping"]

In [4]:
start_time, start_mem = start_profiling()

In [5]:
image_set_loader = ImageSetLoader(
    image_set_path=image_set_path,
    mask_set_path=mask_set_path,
    anisotropy_spacing=(1, 0.1, 0.1),
    channel_mapping=channel_n_compartment_mapping,
    image_set_name=well_fov,
    mask_key_name=[channel_n_compartment_mapping[compartment]],
    raw_image_key_name=[channel_n_compartment_mapping[channel]],
)

In [6]:
object_loader = ObjectLoader(
    image_set_loader.image_set_dict[channel],
    image_set_loader.image_set_dict[compartment],
    channel,
    compartment,
)

In [7]:
if processor_type == "CPU":
    output_dict = measure_3D_intensity_CPU(object_loader)
else:
    raise ValueError(f"Processor type {processor_type} is not supported. Use 'CPU'.")
final_df = pd.DataFrame(output_dict)
# prepend compartment and channel to column names
final_df = final_df.pivot(
    index=["object_id"],
    columns="feature_name",
    values="value",
).reset_index()
final_df.rename(
    columns={
        col: format_morphology_feature_name(
            compartment=compartment,
            channel=channel,
            feature_type="Intensity",
            measurement=col,
        )
        if col != "object_id"
        else col
        for col in final_df.columns
    },
    inplace=True,
)

final_df.insert(0, "image_set", image_set_loader.image_set_name)

output_file = pathlib.Path(
    output_parent_path
    / f"Intensity_{compartment}_{channel}_{processor_type}_features.parquet"
)
output_file.parent.mkdir(parents=True, exist_ok=True)
final_df.to_parquet(output_file)
final_df.head()

feature_name,image_set,object_id,Cytoplasm_Mito_Intensity_CMI-X,Cytoplasm_Mito_Intensity_CMI-Y,Cytoplasm_Mito_Intensity_CMI-Z,Cytoplasm_Mito_Intensity_IntegratedIntensity,Cytoplasm_Mito_Intensity_IntegratedIntensityEdge,Cytoplasm_Mito_Intensity_LowerQuartileIntensity,Cytoplasm_Mito_Intensity_MassDisplacement,Cytoplasm_Mito_Intensity_MaxIntensity,...,Cytoplasm_Mito_Intensity_MaxZ,Cytoplasm_Mito_Intensity_MeanAbsoluteDeviationIntensity,Cytoplasm_Mito_Intensity_MeanIntensity,Cytoplasm_Mito_Intensity_MeanIntensityEdge,Cytoplasm_Mito_Intensity_MedianIntensity,Cytoplasm_Mito_Intensity_MinIntensity,Cytoplasm_Mito_Intensity_MinIntensityEdge,Cytoplasm_Mito_Intensity_StdIntensity,Cytoplasm_Mito_Intensity_StdIntensityEdge,Cytoplasm_Mito_Intensity_UpperQuartileIntensity
0,C4-2,257,530.237671,573.746033,12.328723,6.185237e+08,53235496.0,3855.0,1.291685,65535.0,...,5.0,465.765991,4308.678223,2180.888672,4112.0,2313.0,0.0,1148.454956,2419.511230,4626.0
1,C4-2,514,490.213165,222.546417,13.122419,1.771357e+09,83265176.0,3341.0,6.780295,11565.0,...,1.0,662.456665,3890.937500,2027.939575,3855.0,1542.0,0.0,839.855347,2167.928467,4369.0
2,C4-2,771,349.256042,707.603027,11.757418,2.012159e+09,70614608.0,3084.0,3.841076,12850.0,...,10.0,637.879150,3739.808594,1866.481812,3598.0,1542.0,0.0,828.361084,1984.929077,4112.0
3,C4-2,1028,571.081909,820.288879,17.545012,9.391972e+08,63138732.0,3598.0,1.662579,14906.0,...,0.0,501.740448,4048.944580,2086.472168,3855.0,1799.0,0.0,646.105408,2164.613037,4369.0
4,C4-2,1285,578.496704,922.736511,14.535471,3.771600e+09,93463704.0,2827.0,3.259292,14906.0,...,4.0,517.636963,3389.714844,1676.298584,3341.0,1285.0,0.0,680.253845,1768.829834,3855.0


In [8]:
stop_profiling(
    start_time=start_time,
    start_mem=start_mem,
    feature_type="Intensity",
    well_fov=well_fov,
    patient_id=patient,
    channel=channel,
    compartment=compartment,
    CPU_GPU=processor_type,
    output_file_dir=pathlib.Path(
        f"{root_dir}/data/{patient}/extracted_features/run_stats/{well_fov}_{channel}_{compartment}_Intensity_{processor_type}.parquet"
    ),
)


        Memory and time profiling for the run:
        Patient ID: NF0014_T1
        Well and FOV: C4-2
        Feature type: Intensity
        CPU/GPU: CPU
        Peak memory (tracemalloc): 1552.24 MB
        Current memory (tracemalloc): 303.08 MB
        RSS at end: 540.24 MB
        Time elapsed:
        --- 90.14 seconds ---
        --- 1.50 minutes ---
        --- 0.03 hours ---
    


True